# Capstone — Can Behavioral Signals Predict Content Decline?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ErenSnowh/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Vinayak Pandey · [Portfolio](https://allabtme.vercel.app/) · [GitHub](https://github.com/ErenSnowh)

### The FlyRank Case Study

FlyRank manages content optimization across dozens of client portfolios — thousands of pages per client, each with its own search performance trajectory. In large-scale autonomous content operations, pages inevitably experience search decay: rankings slip, impressions drop, and manual triage becomes an operational bottleneck.

Existing production workflows rely on hand-written heuristic flags (such as static health scores and freshness thresholds), which struggle to capture multi-variable interactions in noisy search telemetry. The recurring editorial problem is **triage at scale**: *when a content team has budget to review 50 pages this sprint, which 50 should they pick?*

This notebook is the computational backbone of the deployed research paper. It trains and validates a decision-tree classifier evaluated on strict client-holdout splits (clients unseen during training), comparing against FlyRank's hand-crafted rule baseline. Every table, chart, and claim in the paper is generated from the code below — reproducibly.

> **Deployed paper:** [https://ErenSnowh.github.io/flyrank-ml-internship/](https://ErenSnowh.github.io/flyrank-ml-internship/)


In [1]:
%pip install -q pandas numpy scikit-learn matplotlib

import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

REPO_ROOT = Path(os.getcwd())
if (REPO_ROOT / 'data' / 'raw').exists():
    pass
elif (REPO_ROOT.parent.parent / 'data' / 'raw').exists():
    REPO_ROOT = REPO_ROOT.parent.parent
else:
    raise FileNotFoundError('Cannot find repo root')

RAW_CSV = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
PIPELINE_QUEUE = REPO_ROOT / 'outputs' / 'refresh_queue.csv'
W05_RESULTS = REPO_ROOT / 'work' / 'outputs' / 'w05_model_results.json'
W06_RESULTS = REPO_ROOT / 'work' / 'outputs' / 'w06_validation_audit_results.json'
W07_RECEIPT = REPO_ROOT / 'work' / 'outputs' / 'w07_playbook_receipt.json'
RANDOM_STATE = 42

print(f'Repo root: {REPO_ROOT}')



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\suzum\Downloads\flyrankinternproject\.venv\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Repo root: C:\Users\suzum\Downloads\flyrankinternproject


## 1. Question

**The FlyRank problem:** FlyRank's content teams manage large client portfolios where pages
silently lose search visibility over time. Identifying which pages are declining — before
the client notices — is the difference between proactive optimization and firefighting.

**Research question:** Can behavioral signals from search and analytics data predict which
content pages are experiencing declining impression trends — well enough to prioritize
editorial review?

**Decision supported:** Which pages in a large content portfolio should a content editor
review and potentially refresh first?

| Dimension | Answer |
|---|---|
| **Unit of analysis** | One content page (pseudonymized) |
| **Output** | Ranked queue with scores, reason codes, and action labels |
| **Action a human takes** | Reviews the flagged page, confirms the signal, decides whether to refresh |
| **Cost of a wrong call** | ~2–6 hours wasted editorial time per false positive |
| **Why this matters to FlyRank** | Proactive refresh → better client retention, fewer "why did my page drop?" conversations |


In [2]:
# Load data and model receipts
raw = pd.read_csv(RAW_CSV)
queue = pd.read_csv(PIPELINE_QUEUE)
with open(W05_RESULTS) as f: w05 = json.load(f)
with open(W06_RESULTS) as f: w06 = json.load(f)
with open(W07_RECEIPT) as f: w07 = json.load(f)

print(f'Raw data: {raw.shape}')
print(f'Queue: {queue.shape}')
print(f'Best model: {w05["best_model"]}')


Raw data: (30000, 44)
Queue: (30000, 28)
Best model: decision_tree


## 2. Data

**Source:** FlyRank ML Internship anonymized starter release.
- 30,000 rows x 44 columns, one row per pseudonymized content page
- 32 pseudonymized clients, trailing 90-day metrics
- Label: `is_declining_label` = 1 when `trend_direction == 'down'` (54.2% positive rate)

**Excluded columns:**
- `trend_direction`, `trend_pct` — label sources (leakage)
- `content_id`, `client_id` — pseudonymous IDs (grouping only, never features)
- `provider_used`, `model_used` — LLM provenance (not performance signals)

**No client names, URLs, domains, or raw queries appear anywhere in this work.**

In [3]:
# Data summary
print(f'Total pages: {len(raw):,}')
print(f'Clients: {raw["client_id"].nunique()}')
print(f'Columns (raw): {raw.shape[1]}')
print(f'Label positive rate: {raw["trend_direction"].eq("down").mean():.1%}')
print(f'\nContent types: {raw["content_type"].value_counts().to_dict()}')
print(f'Missing word_count: {raw["word_count"].isna().sum():,} rows')


Total pages: 30,000
Clients: 32
Columns (raw): 44
Label positive rate: 54.2%

Content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}
Missing word_count: 7,699 rows


## 3. Methodology

### Label definition
`is_declining_label = 1` when `trend_direction == 'down'` (impression change < -20%).
This is a proxy for editorial attention priority, not content quality.

### Feature engineering
- 18 numeric + 8 categorical features (one-hot encoded)
- Log-transformed traffic columns, binary indicators
- Blanks filled (numerics → 0, categoricals → 'unknown')

### Baseline
Hand-crafted rule score: 40% visibility + 30% freshness risk + 25% position opportunity + 5% depth gap.

### Models
- Logistic Regression, Decision Tree (depth=5), Random Forest (n=200)

### Validation
- Client-holdout split (~20% of clients held out entirely)
- Leakage checks: zero forbidden columns, injection attack test passed

In [4]:
# Leakage verification (from W06 receipt)
print(f'Forbidden columns leaked: {w06["forbidden_columns_leaked"]}')
print(f'Injection attack test passed: {w06["injection_attack_test_passed"]}')
print(f'Leakage assertion passed: {w06["leakage_assertion_passed"]}')
print(f'\nSplit strategy: {w05["split_strategy"]}')
print(f'Train rows: {w05["train_rows"]:,}')
print(f'Test rows: {w05["test_rows"]:,}')


Forbidden columns leaked: 0
Injection attack test passed: True
Leakage assertion passed: True

Split strategy: client_holdout
Train rows: 27,675
Test rows: 2,325


## 4. Results (vs baseline)

All metrics on the same client-holdout split. Base rate: 39.1%.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Results table
results = w05['results']
print(f'{"Model":<25} {"P@20":>6} {"P@50":>6} {"ROC-AUC":>8} {"Avg Prec":>9}')
print('-' * 60)
for name, metrics in results.items():
    p20 = metrics.get('Precision@20', '-')
    p50 = metrics.get('Precision@50', '-')
    auc = metrics.get('ROC-AUC', '-')
    ap = metrics.get('Avg Precision', '-')
    print(f'{name:<25} {p20:>6} {p50:>6} {auc:>8} {ap:>9}')
print()
print(f'Base rate: {w05["base_rate"]:.1%}')
best = w05['best_model']
best_p50 = results[best]['Precision@50']
bl_p50 = results['baseline_rules']['Precision@50']
print(f'Best model ({best}) lift over baseline: {best_p50/bl_p50:.1f}x')
print(f'Best model ({best}) lift over random: {best_p50/w05["base_rate"]:.1f}x')


Model                       P@20   P@50  ROC-AUC  Avg Prec
------------------------------------------------------------
baseline_rules              0.15   0.24   0.6269    0.4676
logistic_regression         0.35    0.4   0.7003    0.5215
decision_tree                0.8   0.68   0.7415    0.5753
random_forest                0.7   0.68   0.7474    0.6101

Base rate: 39.1%
Best model (decision_tree) lift over baseline: 2.8x
Best model (decision_tree) lift over random: 1.7x


In [6]:
# Model comparison chart
models = ['baseline_rules', 'logistic_regression', 'decision_tree', 'random_forest']
labels = ['Baseline Rules', 'Logistic Regression', 'Decision Tree', 'Random Forest']
p50_vals = [results[m]['Precision@50'] for m in models]
colors = ['#d4d4d4', '#B07AA1', '#426B69', '#4E79A7']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(labels[::-1], p50_vals[::-1], color=colors[::-1])
ax.axvline(x=w05['base_rate'], color='#e74c3c', linestyle='--', label=f'Base rate: {w05["base_rate"]:.2f}')
ax.set_xlabel('Precision@50')
ax.set_title('Precision@50: Model vs Baseline (Client-Holdout Split)')
ax.legend()
for bar, val in zip(bars, p50_vals[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}',
            va='center', fontsize=10)
plt.tight_layout()
plt.show()


C:\Users\suzum\AppData\Local\Temp\ipykernel_25824\2628809627.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Feature importance
top_features = w05['top_features'][:7]
feat_names = [f['feature'] for f in top_features]
feat_imps = [f['importance'] for f in top_features]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(feat_names[::-1], feat_imps[::-1], color='#426B69')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top Feature Importances (Decision Tree)')
for i, v in enumerate(feat_imps[::-1]):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 2 features (days_with_impressions + content_age_days) account for '
      f'{feat_imps[0]+feat_imps[1]:.0%} of model splits.')



Top 2 features (days_with_impressions + content_age_days) account for 68% of model splits.


C:\Users\suzum\AppData\Local\Temp\ipykernel_25824\3737949276.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Limitations

1. **Cross-sectional, not causal.** We observed associations between behavioral signals and
   declining trends. We did NOT test whether refreshing a flagged page reverses the decline.
   An A/B test would be needed for causal claims.

2. **One snapshot in time.** Seasonal effects, algorithm updates, or market shifts may
   change which signals matter.

3. **32 clients only.** Generalization to new client types has not been tested.

4. **No content-quality features.** The model scores behavioral signals, not content quality.

5. **Label is a proxy.** Impression decline ≠ content needs refreshing (seasonality, sunsetting).

6. **False-positive rate in top-50: 32%.** Human review is mandatory before acting.

In [8]:
# Quantified limitations
best_p50 = results[w05['best_model']]['Precision@50']
print(f'Base rate (random precision): {w05["base_rate"]:.1%}')
print(f'Model Precision@50: {best_p50:.1%}')
print(f'False positive rate in top 50: {1-best_p50:.1%}')
print(f'-> Of every 50 pages flagged, ~{int(50*best_p50)} are genuinely declining, '
      f'~{int(50*(1-best_p50))} are not.')


Base rate (random precision): 39.1%
Model Precision@50: 68.0%
False positive rate in top 50: 32.0%
-> Of every 50 pages flagged, ~34 are genuinely declining, ~15 are not.


## 6. Ranked Recommendations

The action playbook (from ML-10) assigns each page a blended score, transparent reason codes,
and one of five action labels. See the full playbook in
[w07_action_playbook.ipynb](https://github.com/ErenSnowh/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb).

In [9]:
# Action distribution from W07
print('=== Action Distribution ===')
for action, count in w07['action_distribution'].items():
    pct = count / w07['total_pages_scored'] * 100
    print(f'  {action:<35} {count:>6,} ({pct:.1f}%)')
print(f'\nHigh-confidence pages: {w07["high_confidence_pages"]:,}')
print(f'Pages needing action: {w07["pages_needing_action"]:,}')
print(f'\nNo-go items: {len(w07["no_go_items"])}')
for item in w07['no_go_items']:
    print(f'  - {item}')


=== Action Distribution ===
  monitor                             13,069 (43.6%)
  refresh                              8,207 (27.4%)
  refresh_and_review_ctr               6,655 (22.2%)
  refresh_and_review_engagement        1,987 (6.6%)
  expand_and_refresh                      82 (0.3%)

High-confidence pages: 3,576
Pages needing action: 16,931

No-go items: 5
  - Auto-publish without human review
  - Delete or redirect pages from model scores alone
  - Set budgets or SLAs from model scores
  - Promise clients that refresh will improve rankings
  - Apply queue to new clients without re-validation


## 7. Artifacts the paper embeds

All artifacts are generated from the code above and committed to the repository.

In [10]:
# Summary of all output artifacts
print('=== Output artifacts ===')
artifacts = [
    ('docs/index.html', 'Deployed research paper (GitHub Pages)'),
    ('work/notebooks/capstone.ipynb', 'This notebook'),
    ('work/outputs/w05_model_results.json', 'Model metrics receipt'),
    ('work/outputs/w06_validation_audit_results.json', 'Validation audit receipt'),
    ('work/outputs/w07_playbook_receipt.json', 'Playbook receipt'),
    ('work/figures/w07_action_confidence_mix.png', 'Action/confidence chart'),
    ('work/figures/w07_reason_code_frequency.png', 'Reason code chart'),
    ('submission/paper_url.txt', 'Deployed paper URL'),
]
for path, desc in artifacts:
    full = REPO_ROOT / path
    exists = full.exists()
    size = full.stat().st_size if exists else 0
    print(f'  {"OK" if exists else "MISSING"}: {path} ({size:,} bytes) — {desc}')


=== Output artifacts ===
  OK: docs/index.html (59,880 bytes) — Deployed research paper (GitHub Pages)
  OK: work/notebooks/capstone.ipynb (33,707 bytes) — This notebook
  OK: work/outputs/w05_model_results.json (1,756 bytes) — Model metrics receipt
  OK: work/outputs/w06_validation_audit_results.json (1,338 bytes) — Validation audit receipt
  OK: work/outputs/w07_playbook_receipt.json (2,048 bytes) — Playbook receipt
  OK: work/figures/w07_action_confidence_mix.png (54,562 bytes) — Action/confidence chart
  OK: work/figures/w07_reason_code_frequency.png (68,919 bytes) — Reason code chart
  OK: submission/paper_url.txt (51 bytes) — Deployed paper URL


---

## 8. 5-Minute Demo Outline (Week-8 Showcase)

A structured, timed walk-through designed for the 5-minute Week-8 Capstone Showcase. Covering the core 5 pillars: **Question · Method · One Key Chart · One Honest Result · One Recommendation**.

---

### [0:00 – 0:45] Pillar 1: The Question & The FlyRank Content Problem
- **The Hook:** "FlyRank builds content as infrastructure across dozens of enterprise clients. But web content decays silently — rankings slip, impressions drop, and editorial teams can only review about 50 pages per sprint out of thousands."
- **The Core Question:** *Can trailing behavioral signals (impressions, clicks, CTR, engagement, content age, position) predict which content pages are experiencing declining impression trends — well enough to prioritize human editorial review?*
- **Operational Stakes:** False positives waste 2–6 hours of editor time each; false negatives leave high-value decaying pages unrefreshed until client retention is impacted.

---

### [0:45 – 1:45] Pillar 2: The Method & Honest Validation
- **The Dataset:** 30,000 pseudonymized pages across 32 clients from FlyRank's trailing 90-day search and analytics warehouse. Completely public-safe: no client names, URLs, domains, or raw queries.
- **Zero Leakage Discipline:** Target sources (`trend_pct`, `trend_direction`) and downstream production flags (`health_score`, `priority_score`) were strictly excluded from feature sets and verified via programmatic injection tests.
- **Client-Holdout Splits:** Instead of a naive random train/test split (which leaks client-level domain authority and topical style), we held out 20% of whole clients. The model is evaluated strictly on client ecosystems it has never seen.
- **Models Compared:** Hand-crafted rule baseline (FlyRank's 4-component heuristic) vs. Logistic Regression, Decision Tree ($depth=5$), and Random Forest.

---

### [1:45 – 2:45] Pillar 3: One Key Chart — Precision@50 Model Comparison
*(Display Figure from Cell 10: Model Comparison)*
- **What this chart shows:** Top-50 precision on the unseen client holdout split across all candidates.
- **The numbers:**
  - **Random Base Rate:** 39.1% (picking pages at random yields ~20 declining pages).
  - **Hand-Crafted Rule Baseline:** 24.0% Precision@50 (only 12 of 50 picks declining — *worse than random*).
  - **Decision Tree ($depth=5$):** **68.0% Precision@50** (34 of 50 picks genuinely declining).
- **The Visual Takeaway:** A simple, interpretable decision tree achieves a **2.8× lift over the rule baseline** and a **1.7× lift over random base rate**.

---

### [2:45 – 3:45] Pillar 4: One Honest Result & The Skeptic's Audit
- **The Primary Result:** Precision@50 of 0.68 demonstrates that behavioral telemetry carries strong directional signal for triage.
- **Top 2 Feature Drivers:** `days_with_impressions` (43.1% Gini importance) and `content_age_days` (24.7%) account for ~68% of all model splits. Declining pages consistently show fragmented search visibility combined with maturing age.
- **The Skeptic's Audit (Honest Limitations):**
  1. *32% False Positive Rate in Top 50:* 16 of every 50 flagged pages are not declining. This is why the system must remain decision-support, never autonomous publishing.
  2. *Observational, not Causal:* We observe association with historical decline; we cannot claim that refreshing a flagged page guarantees ranking recovery without an A/B test.
  3. *Single 90-day Snapshot:* Seasonal dips or site-wide migrations could register as page-level decay.

---

### [3:45 – 4:45] Pillar 5: One Concrete Recommendation & Action Playbook
- **Production Implementation:** Rather than a raw probability score, every page is mapped to:
  1. A blended priority score combining model confidence with business impact.
  2. Human-readable reason codes (e.g., `imp_decay_high`, `freshness_risk`).
  3. A concrete editorial action: `refresh`, `refresh_and_review_ctr`, `refresh_and_review_engagement`, `expand_and_refresh`, or `monitor`.
- **Immediate Sprint 1 Target:** 3,576 high-confidence declining pages queued for immediate editorial review.
- **Strict No-Go Checklist:**
  - Never auto-publish rewritten content without human review.
  - Never delete pages based solely on model decline scores.
  - Never guarantee clients that a refresh will restore top-3 rankings.

---

### [4:45 – 5:00] Wrap-up & Artifacts
- "The full paper is deployed on GitHub Pages, the pipeline receipts and figures are committed, and the notebook runs cleanly top-to-bottom."
- **Deployed Paper:** [https://ErenSnowh.github.io/flyrank-ml-internship/](https://ErenSnowh.github.io/flyrank-ml-internship/)
- **GitHub Repository:** [https://github.com/ErenSnowh/flyrank-ml-internship](https://github.com/ErenSnowh/flyrank-ml-internship)


---

## 9. Two Shareable Cuts (Methodology Social Post & Employer Summary)

### Cut A: Social Post (LinkedIn / X — shareable as-is)

> 🔬 Just published my machine learning research paper from the FlyRank ML Internship:
> *"Can Behavioral Signals Predict Content Decline?"*
>
> **The Problem:** FlyRank manages content portfolios at scale across dozens of clients. But content decays silently in Google search. When an editorial team only has the bandwidth to refresh 50 pages this sprint, which 50 should they choose?
>
> **The Methodology:**
> • 30,000 pseudonymized pages across 32 clients (trailing 90-day search & analytics telemetry).
> • Strict client-holdout validation: evaluated models on clients unseen during training to ensure true generalization.
> • Programmatic zero-leakage assertions: target sources (`trend_direction`, `trend_pct`) and production heuristic outputs were strictly excluded.
>
> **The Key Result:**
> A shallow decision tree achieved **Precision@50 of 0.68** — a **2.8× lift over FlyRank's hand-crafted rule baseline** (0.24) and **1.7× lift over random selection** (0.39).
>
> **What Surprised Me:**
> The hand-crafted rule baseline performed *worse than random guessing* (P@50 = 0.24 vs. 0.39 base rate). Fixed heuristic rules over-indexed on high-impression pages that were actually stable, whereas the learned tree isolated subtle interactions between impression-active days and content maturity.
>
> **The Deliverable:**
> Not a black-box score, but an operational decision-support playbook: 3,576 high-confidence pages mapped to transparent reason codes, concrete editorial action tags (`refresh`, `expand_and_refresh`, `review_ctr`), and a strict "no-go" guardrail list.
>
> 📄 Read the full paper: https://ErenSnowh.github.io/flyrank-ml-internship/
> 💻 Reproducible code & data contract: https://github.com/ErenSnowh/flyrank-ml-internship
>
> Built by Vinayak Pandey (https://allabtme.vercel.app/) during the FlyRank ML Internship.
>
> #MachineLearning #SEO #DataScience #MLOps #SearchAnalytics #AIResearch

---

### Cut B: Employer-Facing Summary (3 sentences — copy-paste ready)

1. **What I built:** I designed and implemented an end-to-end, leakage-free machine learning prioritization pipeline that predicts search visibility decline across a 30,000-page multi-client portfolio using trailing 90-day behavioral and analytics telemetry.
2. **On what data & what it showed:** Evaluated on strict client-holdout splits (holding out 20% of whole clients to prevent identity leakage), my decision-tree model achieved a **Precision@50 of 0.68** — delivering a **2.8× lift over the production hand-crafted rule baseline** (0.24) and correctly surfacing 34 genuinely declining pages in every 50 reviewed.
3. **Why it matters to your team:** The system operationalizes predictions into transparent reason codes, concrete editorial playbooks, and strict deployment guardrails, and is fully documented as a reproducible, [peer-grade research paper](https://ErenSnowh.github.io/flyrank-ml-internship/) with programmatic leakage assertions and zero data leaks.


## Self-check
Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The primary question is crisp: one sentence, one metric, one decision
- [x] Baseline is hand-crafted rules or a sensible heuristic, evaluated on the same split
- [x] Leakage check passed: target-derived and product-flag columns excluded, receipt checked
- [x] Split strategy is defensible: client-holdout split, evaluated on clients the model never saw
- [x] Primary metric matches human capacity: Precision@20 and Precision@50 reported
- [x] Base rate reported alongside every Precision@K
- [x] Top features reported with honest interpretation (association, not cause)
- [x] Skeptic's audit included: what would make these findings wrong, quantified
- [x] Ranked queue has scores, readable reason codes, and action labels
- [x] Action playbook written: what an editor actually does with the output
- [x] Paper deployed on GitHub Pages with working URL
- [x] All 7 required sections present in the paper
- [x] No client names, URLs, domains, or raw queries anywhere in the repo
- [x] ML-12: Paper abstract & intro tie findings to FlyRank's real content problem (case study framing)
- [x] ML-12: 5-minute demo outline structured with Question, Method, One Chart, One Result, One Recommendation
- [x] ML-12: Two shareable cuts (LinkedIn post & 3-sentence employer summary) complete with real URLs
